# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

La estrategia consiste en representar tanto las películas como las preferencias de cada usuario en un espacio vectorial común, y recomendar las películas cuyo vector sea más similar al perfil del usuario.

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

In [2]:
import pandas as pd
import re
import numpy as np
from collections import Counter

# mdoelo
from sentence_transformers import SentenceTransformer


from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

Visualizamos las queries

In [5]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [6]:
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

### Función de limpieza de texto

Aplicamos un pipeline de limpieza estándar: minúsculas, eliminación de puntuación y palabras con números. 


In [7]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

Unificamos las variables relevantes en un texto (todas menos año)

In [8]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". " # probar agregar queries como "quiero ver una de Tarantino" a ver si funciona, sino sacar
    + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

In [9]:
df_pelis.texto.iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. 2001. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

#### El espanglish en ``genre`` y ``keywords``:  
El modelo multilingüe maneja texto en múltiples idiomas, pero fue entrenado con documentos monolingües por separado, no necesariamente con mezcla de idiomas dentro del mismo string

¿Es un problema grave? No. El modelo multilingüe es bastante robusto a esto. Pero sí es algo valioso para mencionar como limitación del corpus.

## Embedding de peliculas

In [10]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Calcular embedding de cada película

No hace falta hacer el promedio para calcularlo, el output del modelo ya es el embedding del documento (pelicula).

In [11]:
pelis_embeddings = model.encode(df_pelis['texto'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Como el dataset de películas y el de embeddings se construyeron en el mismo orden,podemos unirlos directamente por índice

In [12]:
df_pelis = df_pelis.reset_index(drop=True)

pelis_embeddings_df = pd.DataFrame(pelis_embeddings)
pelis_embeddings_df = pelis_embeddings_df.merge(
    df_pelis[['id', 'name']], 
    left_index=True, 
    right_index=True
)

In [13]:
pelis_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,376,377,378,379,380,381,382,383,id,name
0,-0.031084,0.032719,-0.272750,0.105669,0.088466,0.341037,0.109469,0.170736,0.048048,-0.134747,...,-0.053232,0.032655,-0.199530,-0.102709,0.034748,-0.105462,0.243188,0.014966,1,Herida abierta
1,-0.007789,0.076889,-0.077062,0.274105,-0.079885,-0.025508,0.314847,0.059361,0.081432,0.050591,...,0.168026,0.220522,-0.022279,-0.104578,0.325792,-0.082222,-0.040965,-0.080095,2,"Elvira, reina de las tinieblas"
2,-0.017430,0.006371,-0.100607,0.260501,0.053604,0.227207,0.040888,-0.065317,0.184578,-0.041667,...,0.132501,0.318629,-0.022673,0.075589,0.229808,0.034454,0.114800,-0.044366,3,Durmiendo con su enemigo
3,0.137272,0.015918,0.013975,0.060339,-0.133089,0.079739,0.129728,-0.093552,0.063228,0.041110,...,-0.029802,0.355402,0.058575,-0.094542,0.238030,0.101194,0.021751,-0.007258,4,Elizabethtown
4,-0.165777,0.050897,-0.000996,-0.207756,0.106295,-0.271873,-0.016602,0.089710,0.047141,0.133440,...,-0.091462,-0.023140,-0.099317,0.237066,0.205077,-0.302631,0.368318,0.098505,5,Godzilla


## Embeddings de usuarios

Aplicamos la misma función de limpieza que usamos para las sinopsis. La verdad que no hace falta pero debería ser parte del pipeline operativo habitual.

In [14]:
# data_clean_users = pd.DataFrame(usuarios["query"].apply(limpiar_texto))

Embedding de query

In [15]:
queries_embeddings = model.encode(usuarios['query'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
def get_all_historial_embeddings(usuarios_df, pelis_embeddings_df):
    all_embeddings = []
    
    for _, usuario_row in usuarios_df.iterrows():
        historial_embeddings = []
        
        for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
            nombre = usuario_row[col]
            peli_emb = pelis_embeddings_df[pelis_embeddings_df['name'] == nombre]
            if not peli_emb.empty:
                historial_embeddings.append(peli_emb.drop(columns=['id', 'name']).values[0])
            else:
                print(f"Película '{nombre}' no encontrada.")
        
        embedding_promedio = np.mean(historial_embeddings, axis=0)
        all_embeddings.append(embedding_promedio)
    
    return np.array(all_embeddings)

historial_embeddings = get_all_historial_embeddings(usuarios, pelis_embeddings_df)

Película 'Rec' no encontrada.
Película 'El secreto de sus ojos' no encontrada.
Película 'El exorcista' no encontrada.
Película 'Intocable' no encontrada.
Película 'Una mente brillante' no encontrada.
Película 'Paddington' no encontrada.


### Configuración de Ollama

Iniciamos el servidor de Ollama en background y descargamos el modelo a usar.

In [17]:
!pip install ollama

In [18]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Servidor iniciado")

!ollama pull llama3.2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Servidor iniciado



In [19]:
import ollama

### Reescritura de query e inferencia de pesos con LLM (llamada unificada)

Usamos el LLM para hacer dos cosas en **una sola llamada**:

1. **`query_expandida`**: reescribir la query coloquial del usuario como una descripción corta estilo sinopsis cinematográfica, incluyendo género, mood y keywords. Esto reduce la brecha semántica con los textos del dataset (que combinan sinopsis + género + keywords).

2. **`weights`** y **`direccion`**: inferir cuánto peso darle a la query vs. el historial, y si buscar películas similares al perfil o distintas.

**¿Por qué la reescritura mejora los resultados?** El espacio vectorial fue construido con textos que incluyen género y keywords del dataset. Una query coloquial queda lejos de ese espacio; una reescritura con vocabulario cinematográfico queda cerca.

In [20]:
import re

def extraer_json(raw: str) -> str:
    # Remover bloque <think>...</think> que genera deepseek
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    # Por si viene envuelto en ```json ... ```
    raw = raw.removeprefix('```json').removesuffix('```').strip()
    return raw

In [25]:
import json as json_lib

# Mapeo fijo de categoría → parámetros
CATEGORIA_PARAMS = {
    'normal':                    {'weights': [0.7, 0.3], 'direction': 'top-5'},
    'historial_positivo':        {'weights': [0.1, 0.9], 'direction': 'top-5'},
    'historial_negativo':        {'weights': [0.1, 0.9], 'direction': 'bottom-5'},
}

def procesar_query_con_llm(query: str) -> dict:
    """
    Clasifica la query en una de 3 categorías y devuelve los parámetros correspondientes:
      - normal: query específica, no hace referencia al historial
      - historial_positivo: el usuario quiere algo típico o personalizado para él
      - historial_negativo: el usuario quiere algo distinto a su costumbre
    """
    prompt = f"""You are part of a movie recommendation system. Classify the following user query into exactly one of these three categories:

- "normal": the user describes what they want to watch specifically (genre, mood, theme, plot). No reference to their viewing history.
- "historial_positivo": the user wants something similar to what they usually watch, or their request is so vague that their history is the best guide.
- "historial_negativo": the user explicitly wants something DIFFERENT from their usual preferences.

User query: "{query}"

Examples:
"Quiero una película donde un hombre se enfrenta a una organización criminal" → "normal"
"No sé, algo que valga la pena" → "historial_positivo"
"Lo de siempre está bien" → "historial_positivo"
"Quiero algo distinto a lo que vengo viendo" → "historial_negativo"
"Sorpréndeme con algo que no elegiría yo" → "historial_negativo"

Respond ONLY with valid JSON, no extra text:
{{"categoria": "normal" | "historial_positivo" | "historial_negativo"}}"""

    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json',
        options={'temperature': 0.1, 'seed': 42}
    )

    raw = response.message.content.strip()

    try:
        parsed = json_lib.loads(raw)
        categoria = parsed.get('categoria', 'normal').strip()
        if categoria not in CATEGORIA_PARAMS:
            print(f"[WARN] Categoría inválida '{categoria}', usando 'normal'")
            categoria = 'normal'
        params = CATEGORIA_PARAMS[categoria]
        return {
            'categoria': categoria,
            'weights': params['weights'],
            'direction': params['direction']
        }
    except Exception as e:
        print(f"[WARN] JSON parse error for query '{query[:40]}...': {e}")
        return {'categoria': 'normal', 'weights': [0.7, 0.3], 'direction': 'top-5'}

In [27]:


# Procesar todas las queries
resultados_llm = []
for query in usuarios['query']:
    res = procesar_query_con_llm(query)
    resultados_llm.append(res)
    print(f"Query:      {query}")
    print(f"Categoría:  {res['categoria']} | Weights: {res['weights']} | Dirección: {res['direction']}")
    print()

parametros_usuarios = [{'weights': r['weights'], 'direction': r['direction']} for r in resultados_llm]

Query:      Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano...
Categoría:  normal | Weights: [0.7, 0.3] | Dirección: top-5

Query:      Busco algo basado en hechos reales sobre corrupción o poder político...
Categoría:  historial_positivo | Weights: [0.1, 0.9] | Dirección: top-5

Query:      Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental...
Categoría:  normal | Weights: [0.7, 0.3] | Dirección: top-5

Query:      Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas...
Categoría:  historial_positivo | Weights: [0.1, 0.9] | Dirección: top-5

Query:      Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime...
Categoría:  normal | Weights: [0.7, 0.3] | Dirección: top-5

Query:      Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada...
Categoría:  normal | Weights: [0.7, 0.3] | Direcci

In [28]:
user_embeddings_dinamico = []

for i, params in enumerate(parametros_usuarios):
    w = params['weights']
    emb = np.average(
        [queries_embeddings[i], historial_embeddings[i]],
        axis=0,
        weights=w
    )
    user_embeddings_dinamico.append(emb)

user_embeddings_dinamico = np.array(user_embeddings_dinamico)

## Recomendaciones

### Dinámico (LLM)

In [34]:
scores_dinamico = cosine_similarity(user_embeddings_dinamico, pelis_embeddings)

top5_indices_dinamico = []
for i, params in enumerate(parametros_usuarios):
    if params['direction'] == 'bottom-5':
        indices = scores_dinamico[i].argsort()[:5]
    else:
        indices = scores_dinamico[i].argsort()[-5:][::-1]
    top5_indices_dinamico.append(indices)

top5_indices_dinamico = np.array(top5_indices_dinamico)  # shape: (14, 5)

## Validación

In [31]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
    # el U10 es ambiguo, la etiqueta debe estar mal
]

### Dinámico (LLM)

Evaluamos el enfoque con pesos dinámicos y dirección inferida. Para usuarios con `direccion='distinto'`, los géneros esperados se invierten (queremos que las recomendaciones sean distintas al historial), por lo que la métrica de precisión se interpreta diferente.

In [36]:
resultados_eval_din = []

for user_idx, row in usuarios.head(9).iterrows():
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    params = parametros_usuarios[user_idx]
    print(f"Pesos: {params['weights']} | Dirección: {params['direction']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")

    generos_recomendados = Counter()
    peliculas_buenas = 0
    peliculas_malas = []

    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_indices_dinamico[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores_dinamico[user_idx, idx]

        generos_str = pelicula['genre'].strip('[]')
        generos_list = [g.strip() for g in generos_str.split(',')]
        generos_pelicula = set(generos_list)
        generos_recomendados.update(generos_list)

        es_buena = bool(generos_pelicula & generos_esperados)
        if es_buena:
            peliculas_buenas += 1
            marker = "✓"
        else:
            peliculas_malas.append(pelicula['name'])
            marker = "✗"

        print(f"  {rank}. [{marker}] {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")

    print(f"\nConteo de Géneros en Recomendaciones:")
    for genero, freq in generos_recomendados.most_common():
        print(f"  {genero}: {freq}")

    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\nMÉTRICAS:")
    print(f"  Recall (géneros):      {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
    print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
    print(f"  F1-Score:              {f1:.1%}")

    if peliculas_malas:
        print(f"\nPelículas problemáticas (sin géneros esperados):")
        for pelicula in peliculas_malas:
            print(f"    - {pelicula}")

    resultados_eval_din.append({
        'Usuario': row['nombre'],
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'Películas Malas': len(peliculas_malas),
        'Dirección': params['direction'],
        'Pesos': str(params['weights'])
    })

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN — DINÁMICO")
print(f"{'='*70}")
df_eval_din = pd.DataFrame(resultados_eval_din)
df_eval_din.to_csv('evaluacion_dinamico.csv', index=False)
print(df_eval_din.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval_din['Recall'].mean():.1%}")
print(f"  Precision: {df_eval_din['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval_din['F1'].mean():.1%}")



Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Pesos: [0.7, 0.3] | Dirección: top-5
Géneros Esperados: terror, drama, suspense

Top-5 Recomendaciones:
  1. [✓] El ente (1983) — 0.6466
     Géneros: drama, terror
  2. [✓] Tránsito (2006) — 0.6377
     Géneros: drama, misterio, suspense
  3. [✓] Femme Fatale (2003) — 0.6356
     Géneros: crimen, drama, misterio
  4. [✗] Scary Movie 5 (2013) — 0.6113
     Géneros: comedia
  5. [✓] Alone in the Dark (2006) — 0.6111
     Géneros: acción, terror, ciencia ficción

Conteo de Géneros en Recomendaciones:
  drama: 3
  terror: 2
  misterio: 2
  suspense: 1
  crimen: 1
  comedia: 1
  acción: 1
  ciencia ficción: 1

MÉTRICAS:
  Recall (géneros):      100.0%  (3/3)
  Precision (películas): 80.0%  (4/5)
  F1-Score:              88.9%

Películas problemáticas (sin géneros esperados):
    - Scary Movie 5

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobre corru

## AGREGAR CONCLUSIONES